# Home Credit Default Risk: Gradient Boosting Ensemble

GCI Global 2026 in-class competition. Application tables only (`train.csv`, `test.csv`).

## Methodology

This notebook predicts the probability of loan default (metric: ROC-AUC) using only the application tables `train.csv` and `test.csv`; no external data is used. The base is the v2 matrix of 105 features (cleaning, financial ratios, EXT_SOURCE statistics, unsupervised peer-group deviations, frequency encodings). Earlier experiments showed that target encoding, native categorical handling, stacking, pseudo-labelling and further Optuna studies gave no gain, so this pipeline keeps the model set simple and only tests small, label-free feature candidates.

1. **Feature candidates tested with paired folds.** Three groups of candidates are built without using the target: (a) ratio and count features that are not duplicates of existing columns, (b) cross-fitted predictions of `EXT_SOURCE_1/2/3` from the remaining columns, together with the residual of each score against its prediction, and (c) count encodings of repeated numeric values. A group is adopted only if it raises the fold AUC of a fast LightGBM by at least `MIN_PAIRED_GAIN` and wins in at least `MIN_FOLD_WINS` of the 5 shared folds. A paired comparison is much less noisy than comparing two absolute AUCs.
2. **Base models.** LightGBM, XGBoost and CatBoost, each trained with early stopping on the validation fold of one shared stratified 5-fold split and averaged over several seeds. The main CatBoost parameters can be tuned with Optuna (TPE sampler) on the shared folds; the tuned set is adopted only if it beats the hand-set parameters on the same folds.
3. **Rank averaging.** Each model's predictions are converted to normalised ranks and blended with equal weights or with Nelder-Mead weights. The optimised weights are used only if they beat equal weights in a nested cross-validation.

Unsupervised features (frequency encodings, peer-group statistics, the EXT_SOURCE predictions and the count encodings) are computed on the feature values of `train.csv` and `test.csv` together. No label is used to build them and no external data is involved.

All models share the same folds, every seed is fixed (`SEED = 42`), and every model/seed is checkpointed so interrupted runs resume where they stopped.

### References

- Ke, G. et al. (2017). LightGBM: A highly efficient gradient boosting decision tree. *NeurIPS 30*.
- Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD*, 785-794.
- Prokhorenkova, L. et al. (2018). CatBoost: unbiased boosting with categorical features. *NeurIPS 31*.
- Akiba, T., Sano, S., Yanase, T., Ohta, T., & Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. *Proceedings of the 25th ACM SIGKDD*, 2623-2631.


### Kaggle notebooks and other sources consulted

- GCI Global 2026 course notebooks `tutorial_feature_engineering.ipynb`, `tutorial_adding_features.ipynb` and `tutorial_hyperparameter_tuning.ipynb`: baseline ratio features, count encoding of a categorical column, and the CatBoost Optuna search space (`rsm`, `bootstrap_type`, `border_count`, `leaf_estimation_iterations`, depth, L2 regularisation) that Cell 7 adapts.
- Original Kaggle competition "Home Credit Default Risk" (<https://www.kaggle.com/c/home-credit-default-risk>), cited in the competition README. Predicting the EXT_SOURCE scores from the other application columns is a technique reported by participants of that competition.
- Files provided by the course (README, tutorial notebooks, `HomeCredit_columns_description.xlsx`). No external data was used at any stage.

## Cell 0 - Environment setup
Repository paths (`repo_root`, `DATA_DIR`), Colab / local detection, imports, plus the seeding, checkpoint and timing helpers.

In [2]:
# CELL 0 - Environment setup: repository paths, Colab / local detection, imports, seeding and checkpoint helpers
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Colab ships with LightGBM / XGBoost / scikit-learn; install the remaining libraries quietly.
    # Clone the repository first and open this notebook from the clone: nothing is read from Google Drive.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost", "optuna", "pyarrow"], check=True)
# Local: run once in a terminal ->  pip install -r requirements.txt

# Imported after the optional Colab installation above.
import hashlib
import json
import random
import time
import warnings
from contextlib import contextmanager

import catboost
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from IPython.display import Markdown, display
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


def find_repo_root() -> Path:
    """Locate the repository root, whether the notebook runs from the root, from notebooks/ or from a Colab clone.

    A folder is the repository root if it contains .git, or both requirements.txt and a notebooks/ folder.
    The search walks up from the working directory; on Colab it also looks inside /content.
    """
    def is_root(p: Path) -> bool:
        return (p / ".git").exists() or ((p / "requirements.txt").is_file() and (p / "notebooks").is_dir())

    start = Path.cwd().resolve()
    for p in (start, *start.parents):
        if is_root(p):
            return p
    if IN_COLAB:                                   # e.g. the working directory is /content and the clone is inside it
        for p in sorted(Path("/content").glob("*")):
            if p.is_dir() and is_root(p):
                return p
    print("[warn] repository root not found: using the working directory")
    return start


repo_root = find_repo_root()
# Data folder (holds input/, output/ and checkpoints/). HC_COMPETITION_DIR overrides it; the default is <repo_root>/data.
DATA_DIR = Path(os.environ.get("HC_COMPETITION_DIR") or (repo_root / "data")).expanduser().resolve()
input_dir = DATA_DIR / "input"
output_dir = DATA_DIR / "output"
ckpt_dir = DATA_DIR / "checkpoints"
reports_dir = repo_root / "reports"        # aggregated tables that are safe to publish


def rel(p) -> str:
    """Path as shown in logs: relative to the repository root, so no local folder names end up in the outputs."""
    p = Path(p).resolve()
    try:
        return p.relative_to(repo_root).as_posix()
    except ValueError:
        return f"<outside repository>/{p.name}"


def find_file(folder: Path, candidates):
    """Return the first existing file among the candidate names (None if none exist)."""
    for name in candidates:
        if (folder / name).exists():
            return folder / name
    return None


train_file = find_file(input_dir, ["train.csv", "application_train.csv"])
test_file = find_file(input_dir, ["test.csv", "application_test.csv"])
sample_sub_file = find_file(input_dir, ["sample_submission.csv"])

print(f"Environment      : {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"repo_root        : {repo_root.name}/")
print(f"DATA_DIR         : {rel(DATA_DIR)}")
for label, f in [("train", train_file), ("test", test_file), ("sample_submission", sample_sub_file)]:
    print(f"{label:<17}: {rel(f) if f else 'MISSING'}")

if train_file is None or test_file is None:
    raise FileNotFoundError(
        f"train.csv / test.csv not found in '{rel(input_dir)}'. See 'Setup & Data' in the README, "
        "or set the environment variable HC_COMPETITION_DIR to the data folder.")

for d in (output_dir, ckpt_dir):
    d.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int) -> None:
    """Fix the Python and NumPy random number generators.

    LightGBM, XGBoost and CatBoost receive their seeds explicitly through their parameters.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


CACHE_TAG = "init"   # overwritten in Cell 4 with a hash of the feature matrix


def cached(name: str, fn):
    """Return a stored result if it exists; otherwise compute it, store it and return it.

    Checkpoints live in `<DATA_DIR>/checkpoints/`, so an interrupted run resumes where it stopped
    (on Colab the session disk is wiped when the runtime resets).
    Files are written to a temporary path and renamed, so an interruption can never leave a
    corrupted checkpoint behind.
    """
    path = ckpt_dir / f"{CACHE_TAG}_{name}.pkl"
    if path.exists():
        try:
            return joblib.load(path)
        except Exception:
            print(f"[cache] {path.name} is corrupted, recomputing")
    obj = fn()
    tmp = path.with_suffix(".tmp")
    joblib.dump(obj, tmp)
    tmp.replace(path)
    return obj


STEP_TIMES = {}   # seconds spent in each major step, reported in Cell 10


@contextmanager
def timed(step: str):
    """Accumulate the wall-clock time of a block under the given step name."""
    t0 = time.time()
    try:
        yield
    finally:
        STEP_TIMES[step] = STEP_TIMES.get(step, 0.0) + time.time() - t0

## Cell 1 - Global configuration
Every tunable setting of the pipeline lives here.

In [3]:
# CELL 1 - Global configuration
SEED = 42
N_FOLDS = 5
MODEL_SEEDS = [42, 43, 44]      # every base model is trained once per seed and the predictions are averaged
USE_GPU = False                 # CPU only (about 10 GB RAM and 8 cores are assumed)

# ---- Feature candidates (Cell 5): paired comparison against the 105-feature base on the shared folds ----
RUN_FEATURE_CANDIDATES = True   # False = skip the search and use the 105-feature base matrix as is
MIN_PAIRED_GAIN = 0.0003        # mean per-fold AUC gain a candidate group needs over the base matrix
MIN_FOLD_WINS = 4               # ... and it must also win in at least this many of the 5 folds

# ---- CatBoost tuning with Optuna (Cell 7) -------------------------------------------------------------
CAT_TUNE_TRIALS = 8             # 0 disables tuning: the hand-set CatBoost parameters of Cell 6 are used
CAT_TUNE_TIMEOUT_MIN = 40       # stop the study after this many minutes even if trials remain
CAT_TUNE_FOLDS = 3              # the study scores only the first 3 shared folds (speed)
CAT_MIN_GAIN = 0.0005           # tuned parameters are adopted only if they beat the hand-set ones by this much

# ---- Training and blending ----------------------------------------------------------------------------
EARLY_STOP = 200                # early-stopping patience (rounds) on every validation fold
BLEND_MIN_GAIN = 1e-5           # Nelder-Mead blend weights are used only if they beat equal weights by this much (nested CV)

# ---- Reference values from earlier versions, used as reproduction checks ------------------------------
EXPECTED_V2_FEATURES = 105      # size of the v2 feature matrix
V2_QUICK_CV_AUC = 0.75709       # v2 quick_cv result
V2_LGB_OOF_AUC = 0.76004        # v2 LightGBM (untuned) 5-fold OOF AUC
V2_ENSEMBLE_OOF_AUC = 0.76187   # v2 weighted soft voting (tuned LGB + XGB + CatBoost)

# HC_SMOKE_TEST=1 shrinks the boosting rounds so the whole notebook can be dry-run in minutes on tiny data.
SMOKE_TEST = os.environ.get("HC_SMOKE_TEST") == "1"
MAX_ROUNDS = 200 if SMOKE_TEST else 10_000   # cap on boosting rounds (early stopping decides in practice)
LR_SCALE = 5.0 if SMOKE_TEST else 1.0        # larger learning rates keep the dry run short
if SMOKE_TEST:
    MODEL_SEEDS = MODEL_SEEDS[:1]
    CAT_TUNE_TRIALS = min(CAT_TUNE_TRIALS, 2)

assert len(MODEL_SEEDS) >= 1 and len(set(MODEL_SEEDS)) == len(MODEL_SEEDS), "MODEL_SEEDS must be unique"
assert 1 <= CAT_TUNE_FOLDS <= N_FOLDS

seed_everything(SEED)
print(f"SEED = {SEED} | folds = {N_FOLDS} | model seeds = {MODEL_SEEDS}")
print(f"pandas {pd.__version__} | lightgbm {lgb.__version__} | xgboost {xgb.__version__} | "
      f"catboost {catboost.__version__}" + (" | SMOKE TEST MODE" if SMOKE_TEST else ""))

SEED = 42 | folds = 5 | model seeds = [42, 43, 44]
pandas 3.0.6 | lightgbm 4.7.0 | xgboost 3.2.0 | catboost 1.2.10


## Cell 2 - Data loading

In [4]:
# CELL 2 - Data loading (only train.csv / test.csv, per the competition rules)
train = pd.read_csv(train_file)
test = pd.read_csv(test_file)
sample_sub = pd.read_csv(sample_sub_file) if sample_sub_file else None

y = train["TARGET"].astype(int).values
print(f"train: {train.shape} | test: {test.shape} | default rate: {y.mean():.4f}")
assert train.drop(columns="TARGET").columns.tolist() == test.columns.tolist(), "train/test columns differ"

train: (171202, 34) | test: (61500, 33) | default rate: 0.0807


## Cell 3 - Base feature engineering (v2, unchanged)
The 105-feature v2 matrix is rebuilt as is, or loaded from the parquet cache when it already exists.

In [5]:
# CELL 3 - Base feature engineering (the v2 matrix, unchanged) + parquet cache
# The base matrix is not modified here; the optional new features are tested in Cell 5.
# Set FORCE_REBUILD = True after you change anything inside build_features().
FORCE_REBUILD = False

try:
    import pyarrow  # required by DataFrame.to_parquet / read_parquet
except ImportError as e:
    raise ImportError("pyarrow is missing. Run:  pip install -r requirements.txt") from e

CACHE_X = output_dir / "X_cache.parquet"
CACHE_XT = output_dir / "X_test_cache.parquet"
CACHE_Y = output_dir / "y_cache.npy"
CACHE_IDS = output_dir / "test_ids_cache.npy"
CACHE_CAT = output_dir / "cat_labels_cache.parquet"        # original category labels, train rows then test rows


def safe_div(num: pd.Series, den: pd.Series) -> pd.Series:
    """Element-wise division returning NaN (not inf) when the denominator is 0."""
    return num / den.where(den != 0)


def build_features(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """Build the v2 feature matrices for train and test from the application table only.

    Train and test are concatenated so that frequency encodings and peer-group statistics are
    computed consistently. TARGET is never touched here, so there is no label leakage.
    Gradient-boosting libraries handle NaN natively, so missing values stay NaN
    (plus explicit missing-value counts) instead of being median-imputed.

    Returns the label-encoded matrices (as in v2) and the ORIGINAL category labels of the
    categorical columns (cached for reference; the models use the label-encoded matrices).
    """
    n_train = len(train_df)
    df = pd.concat([train_df.drop(columns="TARGET"), test_df], axis=0, ignore_index=True)

    # ---- 1. Cleaning of known anomalies -------------------------------------
    df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(np.int8)  # 365243 = sentinel
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
    df.loc[df["OWN_CAR_AGE"] >= 60, "OWN_CAR_AGE"] = np.nan                      # EDA finding
    cat_cols_raw = df.select_dtypes(exclude="number").columns.tolist()
    df[cat_cols_raw] = df[cat_cols_raw].replace("XNA", np.nan)                   # XNA = missing
    df["N_MISSING"] = df.isna().sum(axis=1)

    # ---- 2. DAYS_* -> positive years ----------------------------------------
    df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25
    df["YEARS_EMPLOYED"] = -df["DAYS_EMPLOYED"] / 365.25
    df["YEARS_REGISTRATION"] = -df["DAYS_REGISTRATION"] / 365.25
    df["YEARS_ID_PUBLISH"] = -df["DAYS_ID_PUBLISH"] / 365.25
    df["YEARS_LAST_PHONE_CHANGE"] = -df["DAYS_LAST_PHONE_CHANGE"] / 365.25
    df.drop(columns=["DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION",
                     "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE"], inplace=True)

    # ---- 3. Key financial ratios --------------------------------------------
    df["CREDIT_INCOME_RATIO"] = safe_div(df["AMT_CREDIT"], df["AMT_INCOME_TOTAL"])
    df["ANNUITY_INCOME_RATIO"] = safe_div(df["AMT_ANNUITY"], df["AMT_INCOME_TOTAL"])
    df["CREDIT_TERM"] = safe_div(df["AMT_CREDIT"], df["AMT_ANNUITY"])            # approx. months
    df["ANNUITY_CREDIT_RATIO"] = safe_div(df["AMT_ANNUITY"], df["AMT_CREDIT"])
    df["DAYS_EMPLOYED_RATIO"] = safe_div(df["YEARS_EMPLOYED"], df["AGE_YEARS"])
    df["GOODS_CREDIT_RATIO"] = safe_div(df["AMT_GOODS_PRICE"], df["AMT_CREDIT"])
    df["CREDIT_MINUS_GOODS"] = df["AMT_CREDIT"] - df["AMT_GOODS_PRICE"]
    df["INCOME_PER_PERSON"] = safe_div(df["AMT_INCOME_TOTAL"], df["CNT_FAM_MEMBERS"])
    df["ANNUITY_PER_PERSON"] = safe_div(df["AMT_ANNUITY"], df["CNT_FAM_MEMBERS"])
    df["CHILDREN_RATIO"] = safe_div(df["CNT_CHILDREN"], df["CNT_FAM_MEMBERS"])
    df["CAR_AGE_RATIO"] = safe_div(df["OWN_CAR_AGE"], df["AGE_YEARS"])
    df["PHONE_CHANGE_RATIO"] = safe_div(df["YEARS_LAST_PHONE_CHANGE"], df["AGE_YEARS"])
    df["REGISTRATION_RATIO"] = safe_div(df["YEARS_REGISTRATION"], df["AGE_YEARS"])
    bureau_req = ["AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_MON",
                  "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_YEAR"]
    df["BUREAU_REQ_SUM"] = df[bureau_req].sum(axis=1, min_count=1)

    # ---- 4. EXT_SOURCE statistics and interactions (strongest raw signals) ----
    ext = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    df["EXT_MEAN"] = df[ext].mean(axis=1)
    df["EXT_STD"] = df[ext].std(axis=1)
    df["EXT_MIN"] = df[ext].min(axis=1)
    df["EXT_MAX"] = df[ext].max(axis=1)
    df["EXT_PROD"] = df[ext].prod(axis=1, min_count=1)
    df["EXT_MISSING_N"] = df[ext].isna().sum(axis=1)
    df["EXT1_X_EXT2"] = df["EXT_SOURCE_1"] * df["EXT_SOURCE_2"]
    df["EXT1_X_EXT3"] = df["EXT_SOURCE_1"] * df["EXT_SOURCE_3"]
    df["EXT2_X_EXT3"] = df["EXT_SOURCE_2"] * df["EXT_SOURCE_3"]
    df["EXT_MEAN_X_AGE"] = df["EXT_MEAN"] * df["AGE_YEARS"]
    df["EXT_MEAN_X_EMPLOYED"] = df["EXT_MEAN"] * df["YEARS_EMPLOYED"]
    df["CREDIT_PER_EXT"] = safe_div(df["AMT_CREDIT"], df["EXT_MEAN"])
    df["ANNUITY_PER_EXT"] = safe_div(df["AMT_ANNUITY"], df["EXT_MEAN"])

    # ---- 5. Peer-group aggregations (unsupervised: deviation and z-score vs. peers) ----
    group_defs = [["ORGANIZATION_TYPE"], ["OCCUPATION_TYPE"],
                  ["NAME_EDUCATION_TYPE", "NAME_INCOME_TYPE"],
                  ["OCCUPATION_TYPE", "NAME_EDUCATION_TYPE"]]
    agg_targets = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "EXT_MEAN"]
    for keys in group_defs:
        tag = "_".join(keys)
        grp = df.groupby(keys, dropna=False)
        df[f"CNT_GRP_{tag}"] = grp["SK_ID_CURR"].transform("size")
        for col in agg_targets:
            g_mean = grp[col].transform("mean")
            g_std = grp[col].transform("std")
            df[f"{col}_DIFF_GRP_{tag}"] = df[col] - g_mean
            df[f"{col}_Z_GRP_{tag}"] = (df[col] - g_mean) / g_std.where(g_std > 0)

    # ---- 6. Categorical encoding: frequency + label (ordinal) ----------------
    cat_cols_raw = df.select_dtypes(exclude="number").columns.tolist()
    # Keep the ORIGINAL labels before they are overwritten by integer codes (needed for TE / native handling).
    cat_labels = df[cat_cols_raw].astype(object).copy()
    for c in cat_cols_raw:
        df[f"{c}_FREQ"] = df[c].map(df[c].value_counts(normalize=True))   # frequency encoding
        df[c] = pd.factorize(df[c], sort=True)[0]                          # label encoding (NaN -> -1)

    # ---- 7. Final clean-up and memory optimisation ---------------------------
    df = df.replace([np.inf, -np.inf], np.nan)
    feats = [c for c in df.columns if c != "SK_ID_CURR"]
    df[feats] = df[feats].astype("float32")

    X_tr = df.iloc[:n_train][feats].reset_index(drop=True)
    X_te = df.iloc[n_train:][feats].reset_index(drop=True)
    return X_tr, X_te, cat_labels


with timed("Cell 3 - features"):
    if all(p.exists() for p in (CACHE_X, CACHE_XT, CACHE_Y, CACHE_IDS, CACHE_CAT)) and not FORCE_REBUILD:
        # Fast path: reuse the cached matrices (seconds instead of minutes)
        X = pd.read_parquet(CACHE_X)
        X_test = pd.read_parquet(CACHE_XT)
        y = np.load(CACHE_Y)
        cat_labels_all = pd.read_parquet(CACHE_CAT).astype(object)
        print(f"[cache] loaded features from {rel(output_dir)}")
    else:
        t0 = time.time()
        X, X_test, cat_labels_all = build_features(train, test)
        y = train["TARGET"].astype(int).values
        X.to_parquet(CACHE_X)
        X_test.to_parquet(CACHE_XT)
        np.save(CACHE_Y, y)
        np.save(CACHE_IDS, test["SK_ID_CURR"].values)     # test IDs, handy for building the submission later
        cat_labels_all.to_parquet(CACHE_CAT)
        print(f"[build] features built and cached in {time.time() - t0:.0f}s")

cat_cols = cat_labels_all.columns.tolist()   # categorical columns (already integer-coded inside X)
assert len(X) == len(y) and list(X.columns) == list(X_test.columns)
assert set(cat_cols) <= set(X.columns), "categorical columns must be part of the feature matrix"
print(f"X: {X.shape} | X_test: {X_test.shape} | memory: {X.memory_usage().sum() / 1e6:.1f} MB")
print(f"categorical columns ({len(cat_cols)}): {cat_cols}")
if X.shape[1] != EXPECTED_V2_FEATURES:
    print(f"[warn] expected {EXPECTED_V2_FEATURES} v2 features but found {X.shape[1]}")

[cache] loaded features from data/output
X: (171202, 105) | X_test: (61500, 105) | memory: 71.9 MB
categorical columns (8): ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE']


## Cell 4 - Shared folds + fast experiment loop
One stratified split is shared by every model. `quick_cv()` confirms that the rebuilt matrix reproduces the v2 result and keeps the per-fold AUCs for the paired comparisons of Cell 5.

In [6]:
# CELL 4 - Shared CV folds + checkpoint tag + fast reproduction check (quick_cv)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))   # identical folds for every experiment and every model

# Every validation row appears in exactly one fold, and no fold shares rows between train and validation.
assert np.array_equal(np.sort(np.concatenate([va for _, va in folds])), np.arange(len(y)))
assert all(np.intersect1d(tr, va).size == 0 for tr, va in folds)


def feature_tag(X_, prefix="v5"):
    """Short hash of a feature matrix (column names plus a sample of values).

    It prefixes every checkpoint file, so if the feature set changes all old checkpoints are ignored.
    """
    h = hashlib.md5(",".join(map(str, X_.columns)).encode())
    sample = X_.iloc[::97].to_numpy(dtype="float64")
    h.update(np.ascontiguousarray(np.nan_to_num(sample, nan=-1e9)).tobytes())
    return f"{prefix}_{h.hexdigest()[:8]}"


CACHE_TAG = feature_tag(X)
print(f"CACHE_TAG = {CACHE_TAG}")

FAST_PARAMS = dict(
    objective="binary", n_estimators=min(2000, MAX_ROUNDS), learning_rate=0.08, num_leaves=31, max_depth=7,
    min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
    reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, n_jobs=-1, verbosity=-1,
)
QUICK_LOG = []     # running log of quick experiments
QUICK_FOLDS = {}   # experiment label -> array with the AUC of every fold (used for paired comparisons)


def quick_cv(X_, label, params=FAST_PARAMS):
    """Fast 5-fold LightGBM (high learning rate) to compare feature sets. Returns the mean fold AUC."""
    t0 = time.time()
    oof = np.zeros(len(X_))
    scores = []
    for tr, va in folds:
        m = lgb.LGBMClassifier(**params)
        m.fit(X_.iloc[tr], y[tr], eval_set=[(X_.iloc[va], y[va])], eval_metric="auc",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        p = m.predict_proba(X_.iloc[va])[:, 1]
        oof[va] = p
        scores.append(roc_auc_score(y[va], p))
    QUICK_FOLDS[label] = np.array(scores)
    QUICK_LOG.append({"experiment": label, "mean_auc": np.mean(scores), "std_auc": np.std(scores),
                      "oof_auc": roc_auc_score(y, oof), "n_features": X_.shape[1],
                      "seconds": round(time.time() - t0)})
    print(f"[{label}] AUC = {np.mean(scores):.5f} +/- {np.std(scores):.5f} | "
          f"{X_.shape[1]} features | {time.time() - t0:.0f}s")
    return float(np.mean(scores))


# Reproduction check: the rebuilt v2 matrix must give (about) the same quick_cv AUC that v2 reported.
BASE_LABEL = "v2 features (reproduction check)"
with timed("Cell 4 - folds + quick_cv"):
    quick_auc = quick_cv(X, BASE_LABEL)
if not SMOKE_TEST and abs(quick_auc - V2_QUICK_CV_AUC) > 0.003:
    print(f"[warn] quick_cv AUC {quick_auc:.5f} differs from the v2 value {V2_QUICK_CV_AUC:.5f}: "
          f"the feature matrix may not match v2 (library versions or a stale cache?)")

CACHE_TAG = v5_1759c257
[v2 features (reproduction check)] AUC = 0.75709 +/- 0.00329 | 105 features | 30s


## Cell 5 - Feature candidates with a paired-fold gate

Three groups of label-free candidates are added **one at a time** to the untouched 105-feature base. A group is kept only if it beats the base fold by fold (`MIN_PAIRED_GAIN`, `MIN_FOLD_WINS`). No target value is used to build any candidate, so they cannot leak the label; the EXT_SOURCE predictions are cross-fitted so that a row never sees its own value.

In [7]:
# CELL 5 - Label-free feature candidates, accepted or rejected with a paired-fold gate
from sklearn.model_selection import KFold

# Untouched 105-feature matrix: every candidate group is added to THIS matrix and compared against it.
X_base = X.reset_index(drop=True).copy()
X_test_base = X_test[X.columns].reset_index(drop=True).copy()
base_fold_scores = QUICK_FOLDS[BASE_LABEL]

EPS = 1e-5          # added to every denominator (Home Credit amounts are large)
SENTINEL = 365243   # DAYS_EMPLOYED placeholder meaning "not employed"

# Candidates that only re-express columns the base matrix already holds. Tree models are invariant to
# monotone transforms, so these add no information; they only dilute colsample_bytree with clones.
REDUNDANT_FE = {
    "EXT_1x2", "EXT_2x3", "EXT_1x3",                                                          # products of EXT scores
    "CREDIT_ANNUITY", "CREDIT_GOODS", "ANNUITY_INCOME", "CREDIT_INCOME", "PAYMENT_RATE",      # amount ratios
    "EMPLOYED_YEARS", "EMPLOYED_TO_BIRTH", "REGISTRATION_TO_BIRTH", "PHONE_CHANGE_TO_BIRTH",  # timeline ratios
}


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Return a NEW DataFrame with ratio / count features appended. Every column access is guarded."""
    out = df.copy()
    feats = {}

    def _num(frame, col):
        return pd.to_numeric(frame[col], errors="coerce") if col in frame.columns else None

    def _get(name):
        return feats[name] if name in feats else _num(out, name)

    def _add(name, values):
        if values is not None and name not in out.columns and name not in feats:
            feats[name] = values

    def _ratio(num, den):
        return None if num is None or den is None else num / (den + EPS)

    def _mul(a, b):
        return None if a is None or b is None else a * b

    def _days(days_col, years_col):
        """DAYS_* column if present, otherwise rebuilt from the matching YEARS_* column."""
        s = _num(out, days_col)
        if s is not None:
            return s
        yrs = _num(out, years_col)
        return None if yrs is None else -yrs * 365.25

    # DAYS_EMPLOYED sentinel (only relevant when the raw column is still present)
    if "DAYS_EMPLOYED" in out.columns:
        de_raw = pd.to_numeric(out["DAYS_EMPLOYED"], errors="coerce")
        _add("DAYS_EMPLOYED_ANOM", (de_raw == SENTINEL).astype("int8"))
        out["DAYS_EMPLOYED"] = de_raw.mask(de_raw == SENTINEL)

    # EXT_SOURCE aggregations and pairwise products / ratios (need at least 2 of the 3 scores)
    ext_names = [c for c in ("EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3") if c in out.columns]
    if len(ext_names) >= 2:
        ext = out[ext_names].apply(pd.to_numeric, errors="coerce")
        _add("EXT_MEAN", ext.mean(axis=1))
        _add("EXT_STD", ext.std(axis=1))
        _add("EXT_MIN", ext.min(axis=1))
        _add("EXT_MAX", ext.max(axis=1))
        _add("EXT_PROD", ext.prod(axis=1, min_count=1))
        e1, e2, e3 = (_num(out, c) for c in ("EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"))
        _add("EXT_1x2", _mul(e1, e2))
        _add("EXT_1_2", _ratio(e1, e2))
        _add("EXT_2x3", _mul(e2, e3))
        _add("EXT_2_3", _ratio(e2, e3))
        _add("EXT_1x3", _mul(e1, e3))
        _add("EXT_3_1", _ratio(e3, e1))

    # Financial ratios (numerator AND denominator must exist)
    credit, annuity = _get("AMT_CREDIT"), _get("AMT_ANNUITY")
    goods, income = _get("AMT_GOODS_PRICE"), _get("AMT_INCOME_TOTAL")
    fam, kids = _get("CNT_FAM_MEMBERS"), _get("CNT_CHILDREN")
    _add("CREDIT_ANNUITY", _ratio(credit, annuity))
    _add("CREDIT_GOODS", _ratio(credit, goods))
    _add("ANNUITY_INCOME", _ratio(annuity, income))
    _add("CREDIT_INCOME", _ratio(credit, income))
    _add("GOODS_INCOME", _ratio(goods, income))
    _add("PAYMENT_RATE", _ratio(annuity, credit))
    _add("INCOME_PER_PERSON", _ratio(income, fam))
    _add("CHILDREN_RATIO", _ratio(kids, fam))

    # Age / employment timelines
    d_birth = _days("DAYS_BIRTH", "AGE_YEARS")
    d_emp = _days("DAYS_EMPLOYED", "YEARS_EMPLOYED")
    d_reg = _days("DAYS_REGISTRATION", "YEARS_REGISTRATION")
    d_id = _days("DAYS_ID_PUBLISH", "YEARS_ID_PUBLISH")
    d_phone = _days("DAYS_LAST_PHONE_CHANGE", "YEARS_LAST_PHONE_CHANGE")
    if d_birth is not None:
        _add("AGE_YEARS", -d_birth / 365.25)
    if d_emp is not None:
        _add("EMPLOYED_YEARS", -d_emp / 365.25)
    _add("EMPLOYED_TO_BIRTH", _ratio(d_emp, d_birth))
    _add("REGISTRATION_TO_BIRTH", _ratio(d_reg, d_birth))
    _add("ID_PUBLISH_TO_BIRTH", _ratio(d_id, d_birth))
    _add("PHONE_CHANGE_TO_BIRTH", _ratio(d_phone, d_birth))

    # Flag counts
    doc_cols = [c for c in out.columns if str(c).startswith("FLAG_DOCUMENT_")]
    if doc_cols:
        _add("DOC_COUNT", out[doc_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1))
    contact_cols = [c for c in ("FLAG_MOBIL", "FLAG_EMP_PHONE", "FLAG_WORK_PHONE",
                                "FLAG_CONT_MOBILE", "FLAG_PHONE", "FLAG_EMAIL") if c in out.columns]
    if contact_cols:
        _add("CONTACT_COUNT", out[contact_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1))

    # Social circle default ratios
    _add("SOCIAL_DEF_30_RATIO", _ratio(_get("DEF_30_CNT_SOCIAL_CIRCLE"), _get("OBS_30_CNT_SOCIAL_CIRCLE")))
    _add("SOCIAL_DEF_60_RATIO", _ratio(_get("DEF_60_CNT_SOCIAL_CIRCLE"), _get("OBS_60_CNT_SOCIAL_CIRCLE")))

    # Credit-bureau request aggregations
    bureau_cols = [c for c in out.columns if str(c).startswith("AMT_REQ_CREDIT_BUREAU_")]
    if bureau_cols:
        bur = out[bureau_cols].apply(pd.to_numeric, errors="coerce")
        _add("BUREAU_REQ_SUM", bur.sum(axis=1, min_count=1))
        _add("BUREAU_REQ_MAX", bur.max(axis=1))

    if feats:
        new = pd.DataFrame(feats, index=out.index).astype("float64")
        new = new.where(np.isfinite(new)).astype("float32")     # +-inf -> NaN
        out = pd.concat([out, new], axis=1)
    return out


def build_ratio_candidate(Xb, Xt):
    """Group 'ratios': new features from add_features() that are not redundant with the base matrix."""
    fe_tr, fe_te = add_features(Xb), add_features(Xt)
    new_cols = [c for c in fe_tr.columns if c not in Xb.columns and c not in REDUNDANT_FE]
    if not new_cols:
        return None
    return fe_tr[new_cols].reset_index(drop=True), fe_te[new_cols].reset_index(drop=True)


def build_ext_candidate(Xb, Xt, n_splits=5):
    """Group 'ext_pred': cross-fitted predictions of EXT_SOURCE_1/2/3 from the other columns.

    For every score a LightGBM regressor is trained on the rows (train + test, no label involved) where the
    score exists. Rows that have the score receive an out-of-fold prediction, so a row never sees its own
    value; rows where it is missing receive the average prediction of the fold models. Also returns the
    residual (score - prediction) and the mean of the three scores after filling the gaps.
    """
    ext_cols = [c for c in ("EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3") if c in Xb.columns]
    if len(ext_cols) < 2:
        return None
    allx = pd.concat([Xb, Xt], axis=0, ignore_index=True)
    n_tr = len(Xb)
    # Columns derived from the scores (means, products, peer-group deviations ...) or from missing-value
    # counts would leak the regression target into its own inputs, so they are excluded.
    blocked = {c for c in allx.columns if ("EXT" in c and c not in ext_cols) or "MISS" in c.upper()}
    print(f"[ext_pred] regression inputs exclude {len(blocked)} columns derived from the EXT scores or from missing-value counts")
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    n_est = 30 if SMOKE_TEST else 200
    new, filled = {}, []
    for col in ext_cols:
        inputs = [c for c in allx.columns if c != col and c not in blocked]
        observed = allx[col].notna().to_numpy()
        obs_idx, mis_idx = np.flatnonzero(observed), np.flatnonzero(~observed)
        pred = np.zeros(len(allx))
        for fit_pos, hold_pos in kf.split(obs_idx):
            fit_rows, hold_rows = obs_idx[fit_pos], obs_idx[hold_pos]
            model = lgb.LGBMRegressor(
                n_estimators=n_est, learning_rate=0.1, num_leaves=31, min_child_samples=50, subsample=0.8,
                subsample_freq=1, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbosity=-1)
            model.fit(allx.loc[fit_rows, inputs], allx.loc[fit_rows, col])
            pred[hold_rows] = model.predict(allx.loc[hold_rows, inputs])
            if len(mis_idx):
                pred[mis_idx] += model.predict(allx.loc[mis_idx, inputs]) / n_splits
        value = allx[col].to_numpy(dtype="float64")
        tag = col[-1]
        new[f"EXTPRED_{tag}"] = pred
        new[f"EXTRES_{tag}"] = np.where(observed, value - pred, np.nan)
        filled.append(np.where(observed, value, pred))
    new["EXTFILL_MEAN"] = np.mean(filled, axis=0)
    new_df = pd.DataFrame(new).astype("float32")
    return new_df.iloc[:n_tr].reset_index(drop=True), new_df.iloc[n_tr:].reset_index(drop=True)


def build_count_candidate(Xb, Xt):
    """Group 'counts': how often a numeric value (or a credit/goods-price pair) repeats in train + test."""
    allx = pd.concat([Xb, Xt], axis=0, ignore_index=True)
    n_tr = len(Xb)
    new = {}
    for c in ("AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_GOODS_PRICE", "AMT_ANNUITY"):
        if c in allx.columns:
            new[f"CNT_VALUE_{c}"] = allx[c].map(allx[c].value_counts())
    pair = [c for c in ("AMT_CREDIT", "AMT_GOODS_PRICE") if c in allx.columns]
    if len(pair) == 2:
        new["CNT_VALUE_CREDIT_GOODS"] = allx.groupby(pair, dropna=False)[pair[0]].transform("size")
    if not new:
        return None
    new_df = pd.DataFrame(new).astype("float32")
    return new_df.iloc[:n_tr].reset_index(drop=True), new_df.iloc[n_tr:].reset_index(drop=True)


CANDIDATES = {}      # group name -> (train columns, test columns)
adopted_groups = []  # filled by the gate below
candidate_report = pd.DataFrame()

if RUN_FEATURE_CANDIDATES:
    with timed("Cell 5 - feature candidates"):
        for name, builder in (("ratios", build_ratio_candidate), ("ext_pred", build_ext_candidate),
                              ("counts", build_count_candidate)):
            built = builder(X_base, X_test_base)
            if built is None:
                print(f"[candidates] {name}: the required columns are missing, skipped")
                continue
            assert built[0].shape[1] == built[1].shape[1] and list(built[0].columns) == list(built[1].columns)
            assert len(built[0]) == len(X_base) and len(built[1]) == len(X_test_base)
            CANDIDATES[name] = built
            print(f"[candidates] {name}: {built[0].shape[1]} new columns -> {list(built[0].columns)}")

        # Paired gate: each group is added alone to the base matrix and compared fold by fold.
        rows, gains = [], {}
        for name, (tr_new, _) in CANDIDATES.items():
            label = f"base + {name}"
            quick_cv(pd.concat([X_base, tr_new], axis=1), label)
            diff = QUICK_FOLDS[label] - base_fold_scores
            gain, wins = float(diff.mean()), int((diff > 0).sum())
            ok = gain >= MIN_PAIRED_GAIN and wins >= MIN_FOLD_WINS
            gains[name] = gain
            rows.append({"group": name, "new_cols": tr_new.shape[1], "mean_gain": gain, "folds_better": f"{wins}/{N_FOLDS}",
                         "worst_fold": float(diff.min()), "adopted": ok})
            if ok:
                adopted_groups.append(name)
        candidate_report = pd.DataFrame(rows)
        if len(candidate_report):
            display(candidate_report.round(5))

        # Several adopted groups: confirm that the combination still beats the best single group.
        if len(adopted_groups) > 1:
            combo_tr = pd.concat([X_base] + [CANDIDATES[n][0] for n in adopted_groups], axis=1)
            quick_cv(combo_tr, "base + all adopted groups")
            combo_gain = float((QUICK_FOLDS["base + all adopted groups"] - base_fold_scores).mean())
            best_single = max(adopted_groups, key=lambda n: gains[n])
            if combo_gain < gains[best_single]:
                print(f"[candidates] the combination ({combo_gain:+.5f}) is worse than '{best_single}' alone "
                      f"({gains[best_single]:+.5f}): keeping only '{best_single}'")
                adopted_groups = [best_single]

X = pd.concat([X_base] + [CANDIDATES[n][0] for n in adopted_groups], axis=1)
X_test = pd.concat([X_test_base] + [CANDIDATES[n][1] for n in adopted_groups], axis=1)
X_test = X_test[X.columns]
assert X.columns.is_unique and list(X.columns) == list(X_test.columns)
assert len(X) == len(y) == len(X_base) and len(X_test) == len(X_test_base)

CACHE_TAG = feature_tag(X)    # model checkpoints are keyed by the FINAL feature matrix
print(f"\nAdopted groups: {adopted_groups if adopted_groups else 'none (the 105-feature base is kept)'}")
print(f"Final matrices: X {X.shape} | X_test {X_test.shape} | CACHE_TAG = {CACHE_TAG}")

[candidates] ratios: 7 new columns -> ['EXT_1_2', 'EXT_2_3', 'EXT_3_1', 'GOODS_INCOME', 'ID_PUBLISH_TO_BIRTH', 'CONTACT_COUNT', 'BUREAU_REQ_MAX']
[ext_pred] regression inputs exclude 22 columns derived from the EXT scores or from missing-value counts
[candidates] ext_pred: 7 new columns -> ['EXTPRED_1', 'EXTRES_1', 'EXTPRED_2', 'EXTRES_2', 'EXTPRED_3', 'EXTRES_3', 'EXTFILL_MEAN']
[candidates] counts: 5 new columns -> ['CNT_VALUE_AMT_INCOME_TOTAL', 'CNT_VALUE_AMT_CREDIT', 'CNT_VALUE_AMT_GOODS_PRICE', 'CNT_VALUE_AMT_ANNUITY', 'CNT_VALUE_CREDIT_GOODS']
[base + ratios] AUC = 0.75753 +/- 0.00335 | 112 features | 26s
[base + ext_pred] AUC = 0.75920 +/- 0.00277 | 112 features | 24s
[base + counts] AUC = 0.75667 +/- 0.00331 | 110 features | 23s


,group,new_cols,mean_gain,folds_better,worst_fold,adopted
0,ratios,7,0.00044,4/5,-0.00150,True
1,ext_pred,7,0.00211,5/5,0.00122,True
2,counts,5,-0.00042,2/5,-0.00185,False


[base + all adopted groups] AUC = 0.75950 +/- 0.00325 | 119 features | 26s

Adopted groups: ['ratios', 'ext_pred']
Final matrices: X (171202, 119) | X_test (61500, 119) | CACHE_TAG = v5_94e5f4cb


## Cell 6 - Base-model hyper-parameters
Hand-set values carried over from v2 (two earlier Optuna studies gave no gain for LightGBM and XGBoost). CatBoost may be replaced by the tuned set of Cell 7.

In [8]:
# CELL 6 - Base-model hyper-parameters (hand-set; seeds and iteration counts are injected per run in Cell 8)
best_lgb_params = dict(
    objective="binary", n_estimators=MAX_ROUNDS, learning_rate=0.02 * LR_SCALE, num_leaves=31, max_depth=8,
    min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
    reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, n_jobs=-1, verbosity=-1,
    force_row_wise=True, deterministic=True,
)
best_xgb_params = dict(
    n_estimators=MAX_ROUNDS, learning_rate=0.02 * LR_SCALE, max_depth=5, min_child_weight=10,
    subsample=0.8, colsample_bytree=0.5, reg_alpha=0.1, reg_lambda=1.0,
    tree_method="hist", eval_metric="auc", random_state=SEED, n_jobs=-1,
)
best_cat_params = dict(
    learning_rate=0.04 * LR_SCALE, depth=6, l2_leaf_reg=5, eval_metric="AUC",
    random_seed=SEED, verbose=0, thread_count=-1,
)

## Cell 7 - CatBoost tuning with Optuna (optional)

CatBoost is the weakest base model, so it has the most room. The search space follows the course tuning tutorial (`rsm`, `bootstrap_type`, `border_count`, `leaf_estimation_iterations`, depth, L2) but is scored on the **shared folds** instead of a single hold-out split. The hand-set parameters are scored on the same folds first, and the tuned set is adopted only if it beats them (`CAT_MIN_GAIN`, majority of folds). The result is stored in `checkpoints/`; delete that JSON file to tune again. Set `CAT_TUNE_TRIALS = 0` to skip this cell.

In [9]:
# CELL 7 - Optional CatBoost tuning (Optuna, TPE sampler, scored on the shared folds)
from catboost import CatBoostClassifier

CAT_TUNE_FILE = ckpt_dir / f"{CACHE_TAG}_catboost_tuning.json"
cat_tuning_report = {"status": "disabled", "adopted": False}

# Fixed part of every tuning run: the hand-set parameters plus an early-stopping setup.
CAT_TUNE_BASE = {k: v for k, v in best_cat_params.items()
                 if k not in {"iterations", "n_estimators", "early_stopping_rounds", "verbose",
                              "random_seed", "eval_metric", "allow_writing_files"}}
CAT_TUNE_BASE.update(iterations=MAX_ROUNDS, eval_metric="AUC", random_seed=SEED, early_stopping_rounds=100,
                     verbose=0, allow_writing_files=False)


def cat_fold_scores(overrides, n_folds):
    """AUC of a CatBoost model on each of the first n_folds shared folds (early stopping on the fold)."""
    scores = []
    for tr_idx, va_idx in folds[:n_folds]:
        model = CatBoostClassifier(**{**CAT_TUNE_BASE, **overrides})
        model.fit(X.iloc[tr_idx], y[tr_idx], eval_set=(X.iloc[va_idx], y[va_idx]), use_best_model=True)
        scores.append(roc_auc_score(y[va_idx], model.predict_proba(X.iloc[va_idx])[:, 1]))
    return np.array(scores)


if CAT_TUNE_TRIALS <= 0:
    print("[cat-tune] disabled (CAT_TUNE_TRIALS = 0): the hand-set CatBoost parameters are used")
elif CAT_TUNE_FILE.exists():
    cat_tuning_report = json.loads(CAT_TUNE_FILE.read_text())
    print(f"[cat-tune] loaded {CAT_TUNE_FILE.name} (delete it to tune again): {cat_tuning_report.get('status')}")
else:
    try:
        import optuna
    except ImportError:
        cat_tuning_report = {"status": "skipped: optuna is not installed (pip install optuna)", "adopted": False}
        print("[cat-tune] optuna is not installed (pip install optuna); the hand-set parameters are used")
    else:
        n_tune = min(CAT_TUNE_FOLDS, len(folds))
        with timed("Cell 7 - catboost tuning"):
            baseline = cat_fold_scores({}, n_tune)
            print(f"[cat-tune] hand-set parameters: mean AUC {baseline.mean():.5f} on the first {n_tune} folds")
            trial_scores = {}

            def objective(trial):
                bootstrap = trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"])
                p = dict(
                    bootstrap_type=bootstrap,
                    depth=trial.suggest_int("depth", 4, 8),
                    l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 50.0, log=True),
                    rsm=trial.suggest_float("rsm", 0.4, 1.0),
                    random_strength=trial.suggest_float("random_strength", 0.0, 2.0),
                    border_count=trial.suggest_int("border_count", 32, 255),
                    leaf_estimation_iterations=trial.suggest_int("leaf_estimation_iterations", 1, 6),
                )
                if bootstrap == "Bayesian":
                    p["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0.0, 10.0)
                elif bootstrap == "Bernoulli":
                    p["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)
                scores = cat_fold_scores(p, n_tune)
                trial_scores[trial.number] = scores
                print(f"[cat-tune] trial {trial.number}: mean AUC {scores.mean():.5f}")
                return float(scores.mean())

            optuna.logging.set_verbosity(optuna.logging.WARNING)
            study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
            study.optimize(objective, n_trials=CAT_TUNE_TRIALS, timeout=CAT_TUNE_TIMEOUT_MIN * 60)

        if trial_scores:
            best = study.best_trial
            diff = trial_scores[best.number] - baseline
            wins = int((diff > 0).sum())
            adopted = bool(diff.mean() >= CAT_MIN_GAIN and wins >= n_tune // 2 + 1)
            cat_tuning_report = {
                "status": f"{len(trial_scores)} trials on {n_tune} folds", "adopted": adopted,
                "params": dict(best.params), "baseline_mean_auc": float(baseline.mean()),
                "best_mean_auc": float(trial_scores[best.number].mean()), "folds_better": f"{wins}/{n_tune}",
            }
            CAT_TUNE_FILE.write_text(json.dumps(cat_tuning_report, indent=2, default=str))
            print(f"[cat-tune] best trial {best.number}: {cat_tuning_report['best_mean_auc']:.5f} vs "
                  f"{cat_tuning_report['baseline_mean_auc']:.5f} (hand-set) | folds better {wins}/{n_tune} | "
                  f"adopted = {adopted}")
        else:
            cat_tuning_report = {"status": "no trial finished inside the time limit", "adopted": False}
            print("[cat-tune] no trial finished inside the time limit; the hand-set parameters are used")

if cat_tuning_report.get("adopted"):
    best_cat_params = {**best_cat_params, **cat_tuning_report["params"]}
    print(f"[cat-tune] CatBoost parameters used from now on: {best_cat_params}")

[cat-tune] loaded v5_94e5f4cb_catboost_tuning.json (delete it to tune again): 8 trials on 3 folds
[cat-tune] CatBoost parameters used from now on: {'learning_rate': 0.04, 'depth': 5, 'l2_leaf_reg': 3.001230180898044, 'eval_metric': 'AUC', 'random_seed': 42, 'verbose': 0, 'thread_count': -1, 'bootstrap_type': 'MVS', 'rsm': 0.7256176498949491, 'random_strength': 0.2818484499495253, 'border_count': 211, 'leaf_estimation_iterations': 1}


## Cell 8 - Base models (LightGBM, XGBoost, CatBoost)
Shared stratified 5-fold CV with early stopping on every validation fold, repeated for each seed in `MODEL_SEEDS`. OOF and test predictions are averaged over the seeds. Every (model, parameters, seed) run is checkpointed.

In [10]:
# CELL 8 - Base models: 5-fold CV with early stopping, several seeds, checkpointed per (model, params, seed)
import xgboost as xgb

# The test matrix must have exactly the training columns, in the same order.
X_test = X_test[X.columns]
assert not X.select_dtypes(include=["object", "category"]).shape[1], "encode categorical columns first"


def clean_params(params, drop):
    """Remove the keys this pipeline controls itself (iteration count, seed, early stopping ...)."""
    return {k: v for k, v in dict(params).items() if k not in drop}


def lgb_params(seed):
    p = clean_params(best_lgb_params, {
        "n_estimators", "num_iterations", "num_boost_round", "n_iter", "num_trees", "early_stopping_rounds",
        "early_stopping_round", "early_stopping", "random_state", "seed", "verbose", "verbosity"})
    p.update(n_estimators=MAX_ROUNDS, random_state=seed, verbosity=-1)
    p.setdefault("n_jobs", -1)
    p.setdefault("deterministic", True)
    p.setdefault("force_row_wise", True)
    return p


def xgb_params(seed):
    p = clean_params(best_xgb_params, {
        "n_estimators", "num_boost_round", "early_stopping_rounds", "random_state", "seed", "verbosity",
        "eval_metric"})
    p.update(n_estimators=MAX_ROUNDS, random_state=seed, eval_metric="auc", early_stopping_rounds=EARLY_STOP,
             verbosity=0)
    p.setdefault("n_jobs", -1)
    return p


def cat_params(seed):
    p = clean_params(best_cat_params, {
        "n_estimators", "iterations", "num_trees", "early_stopping_rounds", "random_state", "random_seed",
        "verbose", "logging_level", "silent", "eval_metric", "od_type", "od_wait", "allow_writing_files"})
    p.update(iterations=MAX_ROUNDS, random_seed=seed, eval_metric="AUC", early_stopping_rounds=EARLY_STOP,
             verbose=0, allow_writing_files=False)
    p.setdefault("thread_count", -1)
    return p


# Each fit function returns (validation prediction, test prediction, number of trees used)
def fit_lgb(params, X_tr, y_tr, X_va, y_va):
    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="auc",
              callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    n_trees = int(model.best_iteration_)
    return (model.predict_proba(X_va, num_iteration=n_trees)[:, 1],
            model.predict_proba(X_test, num_iteration=n_trees)[:, 1], n_trees)


def fit_xgb(params, X_tr, y_tr, X_va, y_va):
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    n_trees = int(model.best_iteration) + 1            # best_iteration is 0-based
    rng = (0, n_trees)                                  # never use trees past the best one
    return (model.predict_proba(X_va, iteration_range=rng)[:, 1],
            model.predict_proba(X_test, iteration_range=rng)[:, 1], n_trees)


def fit_cat(params, X_tr, y_tr, X_va, y_va):
    model = CatBoostClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
    n_trees = int(model.get_best_iteration()) + 1       # 0-based -> count
    return (model.predict_proba(X_va)[:, 1], model.predict_proba(X_test)[:, 1], n_trees)


def params_tag(params):
    """Short hash of a parameter dict (part of the checkpoint name, so changed params are never reused)."""
    return hashlib.md5(json.dumps(params, sort_keys=True, default=str).encode()).hexdigest()[:8]


def run_cv(name, fit_fn, params, seed):
    """Run the shared folds for one model and seed; return (OOF predictions, fold-averaged test predictions)."""
    def _compute():
        oof, test, iters = np.zeros(len(X)), np.zeros(len(X_test)), []
        for k, (tr_idx, va_idx) in enumerate(folds, 1):
            va_pred, te_pred, n_trees = fit_fn(params, X.iloc[tr_idx], y[tr_idx], X.iloc[va_idx], y[va_idx])
            oof[va_idx] = va_pred
            test += te_pred / N_FOLDS
            iters.append(n_trees)
            print(f"  [{name} seed {seed}] fold {k}/{N_FOLDS}  AUC={roc_auc_score(y[va_idx], va_pred):.5f}  trees={n_trees}")
        return oof, test, iters

    oof, test, iters = cached(f"{name}_{params_tag(params)}_seed{seed}", _compute)
    print(f"[{name}] seed {seed}: OOF AUC = {roc_auc_score(y, oof):.5f} | mean trees = {np.mean(iters):.0f}")
    return oof, test


MODEL_SPECS = (("LightGBM", fit_lgb, lgb_params), ("XGBoost", fit_xgb, xgb_params), ("CatBoost", fit_cat, cat_params))
RESULTS = {}
with timed("Cell 8 - base models"):
    for name, fit_fn, make_params in MODEL_SPECS:
        per_seed = [run_cv(name, fit_fn, make_params(s), s) for s in MODEL_SEEDS]
        oof_mean = np.mean([o for o, _ in per_seed], axis=0)
        test_mean = np.mean([t for _, t in per_seed], axis=0)     # (never name this `test`: it is the raw test DataFrame)
        RESULTS[name] = {"oof": oof_mean, "test": test_mean, "single": roc_auc_score(y, per_seed[0][0]),
                         "avg": roc_auc_score(y, oof_mean)}
        print(f"==> {name}: first seed {RESULTS[name]['single']:.5f} | average of {len(MODEL_SEEDS)} seed(s) {RESULTS[name]['avg']:.5f}\n")

oof_lgb, test_lgb = RESULTS["LightGBM"]["oof"], RESULTS["LightGBM"]["test"]
oof_xgb, test_xgb = RESULTS["XGBoost"]["oof"], RESULTS["XGBoost"]["test"]
oof_cat, test_cat = RESULTS["CatBoost"]["oof"], RESULTS["CatBoost"]["test"]

[LightGBM] seed 42: OOF AUC = 0.76202 | mean trees = 639
[LightGBM] seed 43: OOF AUC = 0.76192 | mean trees = 801
[LightGBM] seed 44: OOF AUC = 0.76223 | mean trees = 799
==> LightGBM: first seed 0.76202 | average of 3 seed(s) 0.76292

[XGBoost] seed 42: OOF AUC = 0.76241 | mean trees = 1132
[XGBoost] seed 43: OOF AUC = 0.76262 | mean trees = 1009
[XGBoost] seed 44: OOF AUC = 0.76241 | mean trees = 952
==> XGBoost: first seed 0.76241 | average of 3 seed(s) 0.76307

[CatBoost] seed 42: OOF AUC = 0.76284 | mean trees = 2005
[CatBoost] seed 43: OOF AUC = 0.76293 | mean trees = 1981
[CatBoost] seed 44: OOF AUC = 0.76291 | mean trees = 2098
==> CatBoost: first seed 0.76284 | average of 3 seed(s) 0.76325



## Cell 9 - Rank-averaging blend
AUC depends only on the ordering of the predictions, so each model is converted to normalised ranks in [0, 1] before blending. Weights are either equal or found with Nelder-Mead; the optimised weights are used only if they beat equal weights in a nested cross-validation (weights fitted on 4 folds, scored on the 5th).

In [11]:
# CELL 9 - Rank averaging of LightGBM, XGBoost and CatBoost (equal weights or Nelder-Mead weights)
from scipy.stats import rankdata

MODEL_NAMES = ["LightGBM", "XGBoost", "CatBoost"]


def norm_rank(p):
    """Average rank (ties share a rank), scaled to [0, 1]."""
    return (rankdata(p, method="average") - 1.0) / (len(p) - 1.0)


R_oof = np.column_stack([norm_rank(p) for p in (oof_lgb, oof_xgb, oof_cat)])
R_test = np.column_stack([norm_rank(p) for p in (test_lgb, test_xgb, test_cat)])


def blend(R, w):
    """Weighted rank average; weights are made non-negative and normalised to sum to 1."""
    w = np.abs(np.asarray(w, dtype=float)) + 1e-12
    return R @ (w / w.sum())


def fit_weights(R, yy):
    """Derivative-free search (Nelder-Mead) for the blend weights that maximise AUC."""
    res = minimize(lambda w: -roc_auc_score(yy, blend(R, w)), x0=np.ones(R.shape[1]) / R.shape[1],
                   method="Nelder-Mead", options={"xatol": 1e-3, "fatol": 1e-7, "maxiter": 200})
    w = np.abs(res.x) + 1e-12
    return w / w.sum()


with timed("Cell 9 - blend"):
    w_equal = np.ones(R_oof.shape[1]) / R_oof.shape[1]
    auc_equal = roc_auc_score(y, blend(R_oof, w_equal))

    # Honest score of the optimised weights: fitted on 4 folds, applied to the 5th
    nested_oof = np.zeros(len(y))
    for tr_idx, va_idx in folds:
        nested_oof[va_idx] = blend(R_oof[va_idx], fit_weights(R_oof[tr_idx], y[tr_idx]))
    auc_nested = roc_auc_score(y, nested_oof)

    w_opt = fit_weights(R_oof, y)                  # final weights: full OOF matrix (3 parameters on ~171k rows)
    use_opt = auc_nested > auc_equal + BLEND_MIN_GAIN
    w_final = w_opt if use_opt else w_equal
    auc_blend = auc_nested if use_opt else auc_equal

print(f"Rank-blend weights ({', '.join(MODEL_NAMES)}):")
print(f"  equal     : {np.round(w_equal, 4)} | OOF AUC = {auc_equal:.5f}")
print(f"  optimised : {np.round(w_opt, 4)} | nested CV AUC = {auc_nested:.5f}")
print(f"  -> using {'optimised' if use_opt else 'equal'} weights | blend AUC = {auc_blend:.5f}")

Rank-blend weights (LightGBM, XGBoost, CatBoost):
  equal     : [0.3333 0.3333 0.3333] | OOF AUC = 0.76392
  optimised : [0.2336 0.2753 0.4911] | nested CV AUC = 0.76393
  -> using equal weights | blend AUC = 0.76392


## Cell 10 - Submission and report
The blended rank score lies in [0, 1] and is written in the row order of `sample_submission.csv` (merged on `SK_ID_CURR`).

In [12]:
# CELL 10 - Submission file and final report
test_blend = blend(R_test, w_final)
pred_df = pd.DataFrame({"SK_ID_CURR": test["SK_ID_CURR"].values, "TARGET": test_blend})
assert len(pred_df) == len(X_test), "X_test rows must be in the same order as test.csv"

if sample_sub is not None:
    submission = sample_sub[["SK_ID_CURR"]].merge(pred_df, on="SK_ID_CURR", how="left")   # sample_submission order
    if not np.array_equal(sample_sub["SK_ID_CURR"].values, pred_df["SK_ID_CURR"].values):
        print("[note] test.csv and sample_submission.csv have different row orders: predictions were re-ordered by ID")
else:
    submission = pred_df
assert len(submission) == len(pred_df) and submission["TARGET"].notna().all(), "IDs do not match the sample submission"
assert submission["TARGET"].between(0, 1).all()

submission_path = output_dir / "submission_v5_rank.csv"
submission.to_csv(submission_path, index=False)

print("=" * 62)
print(f"{'model':<10} {'first seed':>12} {f'avg of {len(MODEL_SEEDS)} seed(s)':>18}")
for name in MODEL_NAMES:
    print(f"{name:<10} {RESULTS[name]['single']:>12.5f} {RESULTS[name]['avg']:>18.5f}")
print(f"{'Rank blend':<10} {'':>12} {auc_blend:>18.5f}  ({'optimised' if use_opt else 'equal'} weights, nested CV)")
print("-" * 62)
print(f"v2 reference: LightGBM {V2_LGB_OOF_AUC:.5f} | tuned ensemble {V2_ENSEMBLE_OOF_AUC:.5f}")
print(f"Features: {X.shape[1]} (adopted groups: {adopted_groups if adopted_groups else 'none'})")
print(f"CatBoost tuning: {cat_tuning_report.get('status')} | adopted = {cat_tuning_report.get('adopted')}")
print(f"Step times (s): { {k: round(v) for k, v in STEP_TIMES.items()} }")
print("=" * 62)
print(f"Saved {rel(submission_path)} ({submission.shape[0]} rows)")

model        first seed   avg of 3 seed(s)
LightGBM        0.76202            0.76292
XGBoost         0.76241            0.76307
CatBoost        0.76284            0.76325
Rank blend                         0.76392  (equal weights, nested CV)
--------------------------------------------------------------
v2 reference: LightGBM 0.76004 | tuned ensemble 0.76187
Features: 119 (adopted groups: ['ratios', 'ext_pred'])
CatBoost tuning: 8 trials on 3 folds | adopted = True
Step times (s): {'Cell 3 - features': 2, 'Cell 4 - folds + quick_cv': 30, 'Cell 5 - feature candidates': 150, 'Cell 8 - base models': 2, 'Cell 9 - blend': 29}
Saved data/output/submission_v5_rank.csv (61500 rows)


In [13]:
# EXPORT FOR TABLEAU AND SQL - paste as the LAST cell, after "CELL 10", and run it in the same session.
# It needs the variables of the finished run (y, folds, RESULTS, R_oof, w_final, blend, train, ...).
# Two folders are written:
#   reports/tableau/              aggregated tables only          -> safe to publish (commit them)
#   <DATA_DIR>/output/local_only/ row-level file for SQL practice -> stays on your computer, never commit it
from sklearn.metrics import roc_curve

export_dir = reports_dir / "tableau"
local_dir = output_dir / "local_only"
export_dir.mkdir(parents=True, exist_ok=True)
local_dir.mkdir(parents=True, exist_ok=True)

oof_blend = blend(R_oof, w_final)                     # out-of-fold rank-blend score of every training row
oof_by_model = {**{name: RESULTS[name]["oof"] for name in MODEL_NAMES}, "Rank blend": oof_blend}
models = list(oof_by_model)

fold_of = np.zeros(len(y), dtype=int)                 # validation fold (1..5) of every training row
for k, (_, va_idx) in enumerate(folds, 1):
    fold_of[va_idx] = k

# 1. AUC per model (average of the seeds) and for the first seed alone
pd.DataFrame({
    "model": models,
    "auc_oof": [roc_auc_score(y, oof_by_model[m]) for m in models],
    "auc_first_seed": [RESULTS[m]["single"] if m in RESULTS else np.nan for m in models],
}).to_csv(export_dir / "model_metrics.csv", index=False)

# 2. AUC of every validation fold (computed on the seed-averaged OOF predictions)
pd.DataFrame([
    {"model": m, "fold": k, "auc": roc_auc_score(y[va_idx], oof_by_model[m][va_idx])}
    for m in models for k, (_, va_idx) in enumerate(folds, 1)
]).to_csv(export_dir / "fold_auc.csv", index=False)

# 3. Blend weights, feature-candidate gate and run time
pd.DataFrame({"model": MODEL_NAMES, "weight": np.asarray(w_final)}).to_csv(export_dir / "model_weights.csv", index=False)
if len(candidate_report):
    candidate_report.to_csv(export_dir / "feature_gate.csv", index=False)
pd.DataFrame({"step": list(STEP_TIMES), "seconds": [round(v, 1) for v in STEP_TIMES.values()]}).to_csv(
    export_dir / "runtime.csv", index=False)

# 4. ROC curve of the blend (300 points)
fpr, tpr, _ = roc_curve(y, oof_blend)
keep = np.linspace(0, len(fpr) - 1, 300).astype(int)
pd.DataFrame({"fpr": fpr[keep], "tpr": tpr[keep]}).to_csv(export_dir / "roc_curve.csv", index=False)

# 5. Reference decile table (to CHECK the one you build with SQL). decile 1 = the riskiest 10% by OOF score
d = pd.DataFrame({"score": oof_blend, "y": y})
d["decile"] = 10 - pd.qcut(d["score"].rank(method="first"), 10, labels=False)
g = d.groupby("decile").agg(n=("y", "size"), defaults=("y", "sum"), avg_score=("score", "mean")).sort_index()
g["default_rate"] = g["defaults"] / g["n"]
g["lift"] = g["default_rate"] / d["y"].mean()
g["cum_pop_pct"] = g["n"].cumsum() / g["n"].sum()
g["cum_defaults_pct"] = g["defaults"].cumsum() / g["defaults"].sum()
g["cum_non_defaults_pct"] = (g["n"] - g["defaults"]).cumsum() / (g["n"] - g["defaults"]).sum()
g.reset_index().to_csv(export_dir / "decile_lift_reference.csv", index=False)

# 6. LOCAL ONLY: one row per training applicant, for SQL practice (join it with train.csv on SK_ID_CURR)
pd.DataFrame({
    "SK_ID_CURR": train["SK_ID_CURR"].values, "fold": fold_of, "TARGET": y, "blend_score": oof_blend,
    "p_lightgbm": RESULTS["LightGBM"]["oof"], "p_xgboost": RESULTS["XGBoost"]["oof"],
    "p_catboost": RESULTS["CatBoost"]["oof"],
}).to_csv(local_dir / "oof_scores_local_only.csv", index=False)

for folder in (export_dir, local_dir):
    for f in sorted(folder.glob("*.csv")):
        print(f"{rel(f):<48} {len(pd.read_csv(f)):>8,} rows")
print("\nPublish only what is inside reports/tableau/. Never publish local_only/.")
